**Parameters**

In [5]:
source_folder = "Files/raw/global_electronics"

StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 7, Finished, Available, Finished, False)

In [9]:
import re, uuid
from pyspark.sql import functions as F

# Spark 4 only accepts a few standard charsets by default; allow Java charsets like windows-1252
spark.conf.set("spark.sql.legacy.javaCharsets", "true")

StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 11, Finished, Available, Finished, False)

**Source configuration**

In [10]:
import re, uuid
from pyspark.sql import functions as F

# One entry per source file: what to read, how to read it, and what columns we expect
SOURCES = [
    {"file": "Sales.csv",          "table": "bronze_sales",          "encoding": "UTF-8",
     "columns": ["Order Number","Line Item","Order Date","Delivery Date","CustomerKey",
                 "StoreKey","ProductKey","Quantity","Currency Code"]},
    {"file": "Customers.csv",      "table": "bronze_customers",      "encoding": "windows-1252",
     "columns": ["CustomerKey","Gender","Name","City","State Code","State","Zip Code",
                 "Country","Continent","Birthday"]},
    {"file": "Products.csv",       "table": "bronze_products",       "encoding": "UTF-8",
     "columns": ["ProductKey","Product Name","Brand","Color","Unit Cost USD","Unit Price USD",
                 "SubcategoryKey","Subcategory","CategoryKey","Category"]},
    {"file": "Stores.csv",         "table": "bronze_stores",         "encoding": "UTF-8",
     "columns": ["StoreKey","Country","State","Square Meters","Open Date"]},
    {"file": "Exchange_Rates.csv", "table": "bronze_exchange_rates", "encoding": "UTF-8",
     "columns": ["Date","Currency","Exchange"]},
]

def to_snake(name):
    """'Order Number' -> 'order_number', 'CustomerKey' -> 'customer_key'"""
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)       # split CamelCase
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()

batch_id = str(uuid.uuid4())
print("Batch:", batch_id)


StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 12, Finished, Available, Finished, False)

Batch: e9c0d057-0787-4db7-a478-fbedbde42bc0


**Check all files are present before loading anything**

In [11]:
files_found = [f.name for f in notebookutils.fs.ls(source_folder)]
missing_files = [s["file"] for s in SOURCES if s["file"] not in files_found]
if missing_files:
    raise FileNotFoundError(f"Missing in {source_folder}: {missing_files}. Found: {files_found}")
print("All source files present")

StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 13, Finished, Available, Finished, False)

All source files present


**Load loop**

In [12]:
results = []
for src in SOURCES:
    path = f"{source_folder}/{src['file']}"

    # 1. Read raw: everything as text, correct encoding, handle quotes inside values (52" LCD)
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .option("encoding", src["encoding"])
          .option("quote", '"').option("escape", '"')
          .csv(path))

    # 2. Schema contract: fail on missing columns, warn on new ones
    missing = [c for c in src["columns"] if c not in df.columns]
    extra   = [c for c in df.columns if c not in src["columns"]]
    if missing:
        raise ValueError(f"{src['file']}: schema contract broken, missing {missing}")
    if extra:
        print(f"WARNING {src['file']}: unexpected new columns {extra}")

    # 3. Independent row count straight from the raw file (lines minus header)
    file_rows = spark.read.text(path).count() - 1

    # 4. Rename columns + add audit columns
    df_bronze = (df.select([F.col(f"`{c}`").alias(to_snake(c)) for c in df.columns]
                           + [F.col("_metadata.file_path").alias("_source_file")])
                   .withColumn("_ingested_at", F.current_timestamp())
                   .withColumn("_batch_id", F.lit(batch_id)))

    # 5. Write: full overwrite, so reruns never duplicate rows
    (df_bronze.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(src["table"]))

    # 6. Reconcile file vs table
    table_rows = spark.table(src["table"]).count()
    status = "PASS" if file_rows == table_rows else "FAIL"
    results.append((src["table"], file_rows, table_rows, status))
    print(f"{src['table']:<24} file={file_rows:>6,}  table={table_rows:>6,}  {status}")

StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 14, Finished, Available, Finished, False)

bronze_sales             file=62,884  table=62,884  PASS
bronze_customers         file=15,266  table=15,266  PASS
bronze_products          file= 2,517  table= 2,517  PASS
bronze_stores            file=    67  table=    67  PASS
bronze_exchange_rates    file=11,215  table=11,215  PASS


**Summary, and fail the run if anything is off**

In [13]:
summary = spark.createDataFrame(results, ["table", "file_rows", "table_rows", "status"])
display(summary)

failed = [r[0] for r in results if r[3] != "PASS"]
if failed:
    raise Exception(f"Reconciliation FAILED for: {failed}")
print("Bronze load complete: all tables reconciled")

StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2b033e66-45ea-490c-99b3-1d4499d7eff4)

Bronze load complete: all tables reconciled


**Validation
1.Encoding check. Add a new cell and run:****

In [14]:
display(spark.table("bronze_customers").filter("city LIKE 'D%sseldorf%'").select("name","city","state").limit(3))

StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6a40477f-43de-4033-a3b8-a013a788fa53)

**2. Rerun test. Run all the cells again. The counts must stay the same. Then run:**

In [15]:
%%sql
DESCRIBE HISTORY bronze_sales

StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 17, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 15 fields>

**3. SQL endpoint check. Open the Lakehouse’s SQL analytics endpoint → New SQL query and run:**

In [17]:
%%sql
SELECT 'sales' AS tbl, COUNT(*) AS row_count FROM dbo.bronze_sales
UNION ALL SELECT 'customers', COUNT(*) FROM dbo.bronze_customers
UNION ALL SELECT 'products',  COUNT(*) FROM dbo.bronze_products
UNION ALL SELECT 'stores',    COUNT(*) FROM dbo.bronze_stores
UNION ALL SELECT 'fx',        COUNT(*) FROM dbo.bronze_exchange_rates;

StatementMeta(, f001aa6a-e36d-496f-820c-2067fc309c93, 19, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 2 fields>